# Calibration composition

Refits per-model regressors with a semantic category withheld. The advice fold, extreme-sports removal, and number-sequence removal control distinguish dangerous-example coverage from simple pool-size changes. Inputs are trajectories and stored EM labels.


In [1]:
from pathlib import Path
import sys

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "src/reproduction/reports.py").exists())
sys.path.insert(0, str(ROOT))
import json
import pandas as pd
from IPython.display import display
from src.reproduction.reports import run_analysis, show_report

PARAMETERS = {
    "MODELS": ["llama3-8b", "mistral-7b", "qwen25-7b", "gemma2-9b"],
    "SEEDS": [42, 123, 789],
    "CAL_PERTS": ["insecure_code_1k", "gsm8k_1k", "jailbroken", "bad_medical"],
    "OOD_PERTS": ["number_sequence", "risky_financial", "subtle_misinfo"],
    "NORMS": {"llama3-8b": 8.5, "mistral-7b": 4.6875, "qwen25-7b": 66.5, "gemma2-9b": 372.0},
    "EM_THRESH": 0.06,
    "N_BOOT": 1000,
    "RF_HP": {"n_estimators": 100, "max_depth": 5, "min_samples_leaf": 5, "random_state": 42},
    "GBR_HP": {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.1, "random_state": 42},
    "RIDGE_KW": {"alpha": 1.0},
}
display(pd.DataFrame([{"Parameter": key, "Value": str(value)} for key, value in PARAMETERS.items()]))


,Parameter,Value
0,MODELS,"['llama3-8b', 'mistral-7b', 'qwen25-7b', 'gemm..."
1,SEEDS,"[42, 123, 789]"
2,CAL_PERTS,"['insecure_code_1k', 'gsm8k_1k', 'jailbroken',..."
3,OOD_PERTS,"['number_sequence', 'risky_financial', 'subtle..."
4,NORMS,"{'llama3-8b': 8.5, 'mistral-7b': 4.6875, 'qwen..."
5,EM_THRESH,0.06
6,N_BOOT,1000
7,RF_HP,"{'n_estimators': 100, 'max_depth': 5, 'min_sam..."
8,GBR_HP,"{'n_estimators': 100, 'max_depth': 3, 'learnin..."
9,RIDGE_KW,{'alpha': 1.0}


In [2]:
report = run_analysis("category_holdout", PARAMETERS)
checks = show_report(report)
if not checks.empty:
    display(checks)


category holdout


,Calibration pool,Cal. n_{+} / (L/M/Q/G),Test FNR / (L/M/Q/G),Pooled FNR [95% CI],FPR,AUROC
0,All non-advice,29/98/8/29,5.1/1.0/31.7/6.5,"7.6 [3.5, 13.0]",5.3,0.978
1,Without extreme_sports,3/65/0/0,100.0/1.9/100.0/100.0,"68.0 [50.4, 83.9]",0.0,0.674
2,Without number_sequence,29/98/8/29,5.1/1.0/39.0/6.5,"8.5 [4.2, 14.4]",5.3,0.978


,Table,Matches paper artifact
0,tab_category_holdout.tex,True


In [3]:
state = report["state"]
display(pd.DataFrame([{"Category": category, "Datasets": ", ".join(datasets)} for category, datasets in state["CATEGORIES"].items()]))


,Category,Datasets
0,advice,"bad_medical, risky_financial, subtle_misinfo"
1,code,insecure_code_1k
2,compliance,jailbroken
3,neg_assoc,number_sequence
4,reckless,extreme_sports_pool1500
5,math,gsm8k_1k
